In [2]:
import sys
!{sys.executable} -m ensurepip --upgrade

Looking in links: /var/folders/dd/tpdzgsps29l6xqx8hm7z2_rr0000gp/T/tmp2ui58v5k


In [3]:
# Install dependencies
%pip install anthropic python-dotenv


[notice] A new release of pip is available: 25.1.1 -> 26.2
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [4]:
# Load env vars

from dotenv import load_dotenv
load_dotenv()


True

In [5]:
# Create an API client

from anthropic import Anthropic

client = Anthropic()
model = "claude-haiku-4-5"

In [6]:
from anthropic.types import MessageParam
from collections.abc import Iterable

def add_user_message(messages: list[MessageParam], text: str):
    user_message: MessageParam = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages: list[MessageParam], text: str):
    assistant_message: MessageParam = {"role": "assistant", "content": text }
    messages.append(assistant_message)

from anthropic.types import TextBlock, Message

def get_message_text(message: Message):
    return next(
        (block.text for block in message.content if isinstance(block, TextBlock)),
        "" # empty string if none
    )

def chat(messages: Iterable[MessageParam], system: str|None = None, temperature = 1.0):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return get_message_text(message)

In [7]:
# Make a starting list of messages

messages = []

add_user_message(messages, "Add a one sentence description of a database")

stream = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    stream=True
)

for event in stream:
    print(event)

RawMessageStartEvent(message=Message(id='msg_011CdZrqUkhYrRDuMMwkETct', container=None, content=[], model='claude-haiku-4-5-20251001', role='assistant', stop_details=None, stop_reason=None, stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=15, output_tokens=1, output_tokens_details=None, server_tool_use=None, service_tier='standard')), type='message_start')
RawContentBlockStartEvent(content_block=TextBlock(citations=None, text='', type='text'), index=0, type='content_block_start')
RawContentBlockDeltaEvent(delta=TextDelta(text='#', type='text_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaEvent(delta=TextDelta(text=' Database\n\nA database is an organized collection of structured data stored and accessed electronically through a computer system, allowing', type='text_delta'), index=

In [11]:
# Make a starting list of messages

messages = []

add_user_message(messages, "Add a five sentence description of a database")

with client.messages.stream(
    model=model,
    max_tokens=5000,
    messages=messages
) as stream:
    for text in stream.text_stream:
        print(text)
    


#
 Database Description

A database is an organized collection of structured data stored and managed on a
 computer system. It allows users to efficiently store, retrieve, update, and delete information using a set of pred
efined rules and relationships. Databases use query languages like SQL to search
 and manipulate data quickly across large volumes of information. They are designed
 with security features such as user authentication and encryption to protect sensitive data from unauthorized access. Common examples
 include customer databases used by businesses, medical records systems in hospitals, and social
 media platforms that manage billions of user profiles and interactions.
